[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anicka-net/nla-at-home/blob/main/notebooks/02_injection_mechanism.ipynb)

# 02 · The Injection Mechanism

### HAAISS workshop — hands-on part 2 of 3

In notebook 01 `describe()` was a black box. Here you **build it** and then **break it two ways** — the two mistakes that cost us the most debugging time.

The whole trick: a language model consumes a sequence of **embedding vectors**. We put a throwaway placeholder token in the prompt, then overwrite *its* embedding with the activation we want the model to describe. The model can't tell the difference between a real token embedding and our smuggled-in vector — as long as the vector looks like the ones it was trained on.

## Setup (same as notebook 01)

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE       = "Qwen/Qwen2.5-7B-Instruct"          # the model whose mind we read
AV_ADAPTER = "anicka/nla-qwen2.5-7b-L20-av-v2"    # the "verbalizer" (activation -> English)
LAYER      = 20                                   # single-layer NLA lives at Qwen layer 20
DEPTH_PCT  = 71                                   # <-- NOT cosmetic. The adapter was TRAINED
                                                  #     at 71% depth (layer 20 of 28). This
                                                  #     number is a CONDITIONING INPUT to the
                                                  #     verbalizer. Notebook 02 lets you feel
                                                  #     what happens when you lie about it.
INJECT_CHAR  = "\u320e"                          # the placeholder token we overwrite: ㈎
INJECT_SCALE = 150.0                              # we normalize the activation's L2 norm TO this

In [ ]:
device = "cuda"
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

# 4-bit so a 7B model + adapters fit a free-Colab T4 (16 GB). fp16 compute:
# the GRPO-sharpened adapter is numerically sensitive, and fp16 on CUDA is a
# tested-safe path (bf16 on Apple MPS collapses it; not our case here).
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

tok  = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                            device_map={"": 0})
model = PeftModel.from_pretrained(base, AV_ADAPTER).eval()   # adapter name = "default"

inject_id = tok.encode(INJECT_CHAR, add_special_tokens=False)
assert len(inject_id) == 1, f"injection char must be ONE token, got {inject_id}"
inject_id = inject_id[0]
print("loaded — base + AV adapter on", next(model.parameters()).device)

In [ ]:
def get_layers(m):
    """Reach the transformer block list through the PEFT + CausalLM wrappers."""
    b = m.base_model.model if hasattr(m, "base_model") else m
    inner = b.model if hasattr(b, "model") else b
    return inner.layers

def read_activation(prompt, layer=LAYER, max_new_tokens=128):
    """Run the model on `prompt` and grab the residual-stream vector at `layer`,
    at the last prompt token (the position that decides the next word)."""
    chat = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(chat, return_tensors="pt").to(device)

    grab = {}
    def hook(mod, inpt, out):
        h = out[0] if isinstance(out, tuple) else out
        if "h" not in grab:                 # FIRST forward pass only — otherwise
            grab["h"] = h[:, -1, :].detach() # every generated token overwrites it
    handle = get_layers(model)[layer].register_forward_hook(hook)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    handle.remove()
    reply = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    return grab["h"].squeeze(0), reply

def normalize_to(v, scale=INJECT_SCALE):
    """Rescale v so its L2 norm equals `scale`. NOT v * scale — see notebook 02."""
    n = v.float().norm().clamp_min(1e-12)
    return v * (scale / n)

def av_prompt(depth_pct):
    return (
        "You are a meticulous AI researcher conducting an important investigation "
        "into activation vectors from a language model. Your overall task is to "
        "describe the semantic content of that activation vector.\n\n"
        "We will pass the vector enclosed in <concept> tags into your context, "
        "along with the network depth where it was extracted. "
        "You must then produce an explanation for the vector, enclosed within "
        "<explanation> tags. The explanation consists of 2-3 text snippets "
        "describing that vector.\n\n"
        f"Here is the vector from depth {depth_pct}% of the network:\n\n"
        f"<concept>{INJECT_CHAR}</concept>\n\n"
        "Please provide an explanation.\n\n"
        "<explanation>")

def describe(activation, depth=DEPTH_PCT, max_new_tokens=120, scale_fn=normalize_to):
    """The whole NLA read: build the prompt, overwrite the placeholder token's
    embedding with the (rescaled) activation, let the model narrate."""
    ids = tok.encode(av_prompt(depth), add_special_tokens=True)
    pos = ids.index(inject_id)
    emb = model.get_input_embeddings()(torch.tensor([ids], device=device)).clone()
    emb[0, pos, :] = scale_fn(activation.to(emb.dtype))
    with torch.no_grad():
        out = model.generate(inputs_embeds=emb, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    seq = out[0]
    gen = seq[len(ids):] if seq.shape[0] > len(ids) else seq  # embeds path returns new-only
    return tok.decode(gen, skip_special_tokens=True).split("</explanation>")[0].strip()

## Build the injection by hand

Forget `describe()` for a moment. Here is the mechanism, unrolled, with the activation of a real prompt.

In [ ]:
activation, _ = read_activation("Explain how a hash map handles collisions.")
print("raw activation L2 norm:", round(activation.float().norm().item(), 1))

# 1) build the prompt text with the placeholder char in it
ids = tok.encode(av_prompt(DEPTH_PCT), add_special_tokens=True)
pos = ids.index(inject_id)          # where the placeholder landed
print("placeholder token id:", inject_id, "at position", pos)

# 2) turn tokens into embeddings
emb = model.get_input_embeddings()(torch.tensor([ids], device=device)).clone()
print("one embedding row looks like:", tuple(emb[0, pos].shape),
      "norm", round(emb[0, pos].float().norm().item(), 1))

### The critical line — normalize, don't multiply

The activation's norm (~130) and a typical token embedding's norm are in the same ballpark, but not identical. The verbalizer was trained on activations whose norm was **set to 150**. So we rescale the vector's length to 150 and keep its direction:

```
normalize_to(v) =  v * (150 / ||v||)      # length becomes exactly 150
```
The classic bug is to write `v * 150` instead — that makes the norm ~19,500, **130× too big**, a vector from a galaxy the model has never seen. Flip the switch and watch it happen.

In [ ]:
# === FLIP THIS ===
BUG = False   # True -> the "multiply by 150" mistake

scale_fn = (lambda v: v * INJECT_SCALE) if BUG else normalize_to
injected = scale_fn(activation.to(emb.dtype))
print("injected norm:", round(injected.float().norm().item(), 1),
      "  (target is 150)")

emb2 = emb.clone()
emb2[0, pos, :] = injected
with torch.no_grad():
    out = model.generate(inputs_embeds=emb2, max_new_tokens=120,
                         do_sample=False, pad_token_id=tok.eos_token_id)
seq = out[0]; gen = seq[len(ids):] if seq.shape[0] > len(ids) else seq
print("\nreadout:\n ", tok.decode(gen, skip_special_tokens=True)
      .split("</explanation>")[0].strip())

Set `BUG = True`, re-run: the norm jumps to ~19,500 and the readout degenerates into repeated / off-topic tokens. That single confusion (`normalize TO` vs `multiply BY`) is mistake #1 in the repo's history-of-pain table.

## The second mistake — the hook that eats itself

`read_activation` guards its hook with `if "h" not in grab`. Here's why. During `generate()` the forward hook fires on **every** generated token, so without the guard `grab["h"]` ends up holding the *last* token's activation — meaningless. Watch the guard matter:

In [ ]:
def read_unguarded(prompt, layer=LAYER):
    chat = tok.apply_chat_template([{"role":"user","content":prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(chat, return_tensors="pt").to(device)
    grab = {}
    def hook(m, i, o):
        h = o[0] if isinstance(o, tuple) else o
        grab["h"] = h[:, -1, :].detach()   # NO guard: overwritten every step
    handle = get_layers(model)[layer].register_forward_hook(hook)
    with torch.no_grad():
        model.generate(**inp, max_new_tokens=40, do_sample=False,
                       pad_token_id=tok.eos_token_id)
    handle.remove()
    return grab["h"].squeeze(0)

p = "Explain how a hash map handles collisions."
good = read_activation(p)[0]
bad  = read_unguarded(p)
print("guarded (prompt state)   ->", describe(good))
print()
print("unguarded (last gen tok) ->", describe(bad))

## The depth number is an input, not a label

`DEPTH_PCT = 71` is fed *into* the verbalizer's prompt. This adapter was trained on layer-20 activations described as coming from **71%** depth. Lie to it and it drifts off-distribution — the caption gets vaguer or wronger even though the vector is identical. (The universal Phi-4 NLA knows every depth; this single-layer Qwen one only knows 71%.)

In [ ]:
act, _ = read_activation("Explain how a hash map handles collisions.")
for d in [71, 40, 96, 10]:
    print(f"depth told = {d:>2}%  ->  {describe(act, depth=d)}")

---
### ✅ Self-check
Expected: `BUG=False` gives an on-topic caption and injected norm ≈ 150; `BUG=True` gives norm ≈ 19,500 and degenerate output; the **guarded** readout is on-topic while the **unguarded** one is not; `depth=71` reads cleaner than the off-distribution depths.

In [ ]:
v, _ = read_activation("Explain how a hash map handles collisions.")
assert abs(normalize_to(v).float().norm().item() - 150) < 1.0
assert (v.float().norm() * 150 > 1000)   # the multiply-bug really is huge
print("self-check: normalize_to lands on 150, multiply-by-150 does not ✓")